# Survivor Machine — is the website's tree real?

On the site you set sex, ticket class and age, and a tree gave you a survival percentage. **Is that number invented, or is it a real calculation?** Let's check it against the actual 1912 data.

## Step 1 — Load the real data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
try:
    df = pd.read_csv(url)
except Exception:
    print("If the download fails, ask your teacher for titanic.csv and upload it to the folder on the left.")
    df = pd.read_csv("titanic.csv")
print("passengers:", len(df), " overall survival rate:", round(df["Survived"].mean(), 3))
df.head()

## Step 2 — Website numbers vs real data

The website widget has these probabilities baked into its tree. Do they actually match the data? Each one should be the survival RATE of the real passengers in that group (survived / total).

In [ ]:
g = df.dropna(subset=["Age"]).copy()

def rate(mask):
    sub = g[mask]
    return (len(sub), round(sub["Survived"].mean(), 2)) if len(sub) else (0, None)

print("                    website   real data (count, rate)")
print("female 1st          0.97     ", rate((g.Sex=="female") & (g.Pclass==1)))
print("female 2nd          0.92     ", rate((g.Sex=="female") & (g.Pclass==2)))
print("female 3rd          0.50     ", rate((g.Sex=="female") & (g.Pclass==3)))
print("boy (<=6) 1-2nd     0.83     ", rate((g.Sex=="male") & (g.Age<=6) & (g.Pclass<=2)))
print("boy (<=6) 3rd       0.36     ", rate((g.Sex=="male") & (g.Age<=6) & (g.Pclass==3)))
print("man (>6) 1st        0.36     ", rate((g.Sex=="male") & (g.Age>6) & (g.Pclass==1)))
print("man (>6) 2-3rd      0.13     ", rate((g.Sex=="male") & (g.Age>6) & (g.Pclass>=2)))

**Observe**: The website numbers are not made up — they are the real proportions from 110-year-old data. So your own verdict on the site was real too: it was the survival rate of passengers who matched your three answers.

## Step 3 — The real tree + feature importance

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

d = df.dropna(subset=["Age"]).copy()
d["Sex_male"] = (d["Sex"] == "male").astype(int)
X = d[["Sex_male", "Pclass", "Age"]]
y = d["Survived"]
clf = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)

plt.figure(figsize=(14, 7))
plot_tree(clf, feature_names=["Sex(male=1)", "Pclass", "Age"],
          class_names=["died", "survived"], filled=True, rounded=True)
plt.show()

print("feature importance:")
for name, imp in zip(X.columns, clf.feature_importances_):
    print(" ", name, round(imp, 3))

**Connect**: The biggest importance is **Sex** — that is why the widget's "cheapest change to flip your fate" was usually changing sex. The tree decided that from data alone, with nobody telling it.

## Step 4 — What the model never measured

On the site, toggling "Brave / Can swim / Lucky" never moved the verdict. Here is why — **there is no such column**. The model only knows what is in the columns.

In [ ]:
print("columns the model could ever see:")
print(list(df.columns))
print()
print("There is no column for 'courage', 'kindness', or 'never gives up'.")

## Questions
- How closely did the website numbers match the real data? Where they differ, why?
- Why is Sex the most important feature — is that a truth about the world, or a truth about who reached the 1912 lifeboats?
- What human things does the model miss entirely?